# Витрина транзакций ТЭ — 7. Два первых пункта анализа

AI-14135. Пункты 1 и 2: сравнение распределений и косинус портретов. Isolation Forest и кластеризация здесь не считаются.

Вход — уже собранный профиль `sandbox_ai.tmp_shestopalov_te_profile`, одна строка на ИНН. Операции заново не читаются. p10 и p90 чека в профиле нет, тесты идут по остальным полям портрета.

Группа меньше 30 клиентов в сравнение распределений и в косинус не входит. Клиенты без кода точки (`mcc_code = не заполнен`) убираются до расчётов: это пустой код, не вид деятельности.

Расчёт лежит в самой тетрадке. Отдельный файл рядом не нужен.


In [ ]:
from IPython.display import display
from rail_connectors.connection import connect

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": "/home/jovyan/test_requests/tech.keytab",
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": "Shestopalov-VYur"},
)
imp._init_connection()


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import jensenshannon

MIN_CLIENTS = 30
COMFORT_CLIENTS = 100
RANDOM_STATE = 23

METRICS = [
    "tx_count",
    "turnover",
    "avg_amount",
    "median_amount",
    "std_amount",
    "p10_amount",
    "p90_amount",
    "avg_interval",
    "median_interval",
    "success_rate",
    "active_days",
    "unique_bins",
    "unique_card_types",
]

# Эти две метрики задают прокси размера. Их убирают из тестов размера только при business_size_is_proxy = 1.
SIZE_BUILT_FROM = {"tx_count", "turnover"}

SLICES = [
    ("MCC", "mcc_code"),
    ("MCC + регион", "slice_mcc_region"),
    ("MCC + размер бизнеса", "slice_mcc_size"),
    ("Группа MCC", "mcc_group"),
    ("Основной ОКВЭД", "okved"),
    ("Основной ОКВЭД, класс", "okved_class"),
]

ASSOCIATIONS = [
    ("Группа MCC x регион", "mcc_group", "region_group"),
    ("Группа MCC x размер бизнеса", "mcc_group", "business_size"),
    ("ОКВЭД класс x регион", "okved_class", "region_group"),
    ("ОКВЭД класс x размер бизнеса", "okved_class", "business_size"),
]

def enough_label(n: int) -> str:
    if n >= COMFORT_CLIENTS:
        return "достаточно"
    if n >= MIN_CLIENTS:
        return "на грани"
    return "мало"


def coverage_table(profile: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for slice_name, col in SLICES:
        if col not in profile.columns:
            continue
        counts = profile[col].fillna("не заполнен").astype(str).value_counts()
        total = int(counts.sum())
        for value, n in counts.items():
            rows.append(
                {
                    "slice_name": slice_name,
                    "slice_value": value,
                    "client_cnt": int(n),
                    "client_share": int(n) / total if total else np.nan,
                    "enough": enough_label(int(n)),
                }
            )
    if not rows:
        return pd.DataFrame(columns=["slice_name", "slice_value", "client_cnt", "client_share", "enough"])
    return pd.DataFrame(rows).sort_values(["slice_name", "client_cnt"], ascending=[True, False])


def bh_adjust(pvalues: pd.Series) -> pd.Series:
    p = pvalues.to_numpy(dtype=float)
    out = np.full(len(p), np.nan)
    mask = np.isfinite(p)
    if mask.sum() == 0:
        return pd.Series(out, index=pvalues.index)
    pv = p[mask]
    n = len(pv)
    order = np.argsort(pv)
    ranked = pv[order]
    adj = ranked * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    placed = np.empty(n)
    placed[order] = adj
    out[np.flatnonzero(mask)] = placed
    return pd.Series(out, index=pvalues.index)


def _hist_js(left: np.ndarray, right: np.ndarray) -> float:
    both = np.concatenate([left, right])
    if both.size < 2 or np.nanmax(both) == np.nanmin(both):
        return np.nan
    bins = np.linspace(np.nanmin(both), np.nanmax(both), 21)
    h_left, _ = np.histogram(left, bins=bins, density=False)
    h_right, _ = np.histogram(right, bins=bins, density=False)
    h_left = h_left.astype(float) + 1e-12
    h_right = h_right.astype(float) + 1e-12
    return float(jensenshannon(h_left, h_right, base=2))


def compare_distributions(profile: pd.DataFrame) -> pd.DataFrame:
    """Каждая группа против остальных клиентов того же разреза."""
    rows = []
    proxy_col = profile.get("business_size_is_proxy")
    size_is_proxy = proxy_col is not None and pd.to_numeric(proxy_col, errors="coerce").fillna(0).eq(1).any()
    for slice_name, col in SLICES:
        if col not in profile.columns:
            continue
        work = profile.copy()
        work["_slice"] = work[col].fillna("не заполнен").astype(str)
        counts = work["_slice"].value_counts()
        usable = [v for v, n in counts.items() if n >= MIN_CLIENTS and (len(work) - n) >= MIN_CLIENTS]
        metrics = [m for m in METRICS if m in work.columns]
        if size_is_proxy and "размер" in slice_name:
            metrics = [m for m in metrics if m not in SIZE_BUILT_FROM]
        for value in usable:
            left_mask = work["_slice"].eq(value)
            for metric in metrics:
                left = pd.to_numeric(work.loc[left_mask, metric], errors="coerce").dropna().to_numpy()
                right = pd.to_numeric(work.loc[~left_mask, metric], errors="coerce").dropna().to_numpy()
                if len(left) < MIN_CLIENTS or len(right) < MIN_CLIENTS:
                    continue
                if np.unique(left).size < 2 and np.unique(right).size < 2:
                    continue
                ks = stats.ks_2samp(left, right)
                mw = stats.mannwhitneyu(left, right, alternative="two-sided")
                rows.append(
                    {
                        "slice_name": slice_name,
                        "slice_value": value,
                        "metric": metric,
                        "n_group": int(len(left)),
                        "n_rest": int(len(right)),
                        "enough": enough_label(int(left_mask.sum())),
                        "ks_stat": float(ks.statistic),
                        "ks_pvalue": float(ks.pvalue),
                        "mannwhitney_pvalue": float(mw.pvalue),
                        "wasserstein": float(stats.wasserstein_distance(left, right)),
                        "jensen_shannon": _hist_js(left, right),
                    }
                )
    if not rows:
        return pd.DataFrame()
    out = pd.DataFrame(rows)
    pieces = []
    for _, part in out.groupby(["slice_name", "metric"], sort=False):
        part = part.copy()
        part["ks_pvalue_bh"] = bh_adjust(part["ks_pvalue"])
        part["mannwhitney_pvalue_bh"] = bh_adjust(part["mannwhitney_pvalue"])
        pieces.append(part)
    return pd.concat(pieces, ignore_index=True)


def association_tests(profile: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for name, left_col, right_col in ASSOCIATIONS:
        if left_col not in profile.columns or right_col not in profile.columns:
            continue
        frame = profile[[left_col, right_col]].copy()
        frame[left_col] = frame[left_col].fillna("не заполнен").astype(str)
        frame[right_col] = frame[right_col].fillna("не заполнен").astype(str)
        frame = frame.loc[~frame[left_col].isin(["не заполнен"])]
        frame = frame.loc[~frame[right_col].isin(["не заполнен", "нет оборота"])]
        keep_left = frame[left_col].value_counts()
        keep_right = frame[right_col].value_counts()
        frame = frame.loc[frame[left_col].isin(keep_left[keep_left >= 5].index)]
        frame = frame.loc[frame[right_col].isin(keep_right[keep_right >= 5].index)]
        if frame[left_col].nunique() < 2 or frame[right_col].nunique() < 2:
            continue
        table = pd.crosstab(frame[left_col], frame[right_col])
        chi2, pvalue, dof, _ = stats.chi2_contingency(table)
        r, c = table.shape
        n = int(table.to_numpy().sum())
        cramer = np.sqrt(chi2 / (n * (min(r, c) - 1))) if n and min(r, c) > 1 else np.nan
        rows.append(
            {
                "association": name,
                "n_clients": n,
                "n_rows": int(r),
                "n_cols": int(c),
                "chi2": float(chi2),
                "dof": int(dof),
                "pvalue": float(pvalue),
                "cramers_v": float(cramer),
            }
        )
    return pd.DataFrame(rows)


def cosine_by_slice(profile: pd.DataFrame) -> pd.DataFrame:
    """Средний профиль группы после z-нормализации, затем косинус между группами."""
    rows = []
    feature_cols = [m for m in METRICS if m in profile.columns]
    if len(feature_cols) < 2:
        return pd.DataFrame()
    usable = profile.copy()
    for col in feature_cols:
        usable[col] = pd.to_numeric(usable[col], errors="coerce")
    usable[feature_cols] = usable[feature_cols].fillna(usable[feature_cols].median())
    vary = [c for c in feature_cols if usable[c].std(ddof=0) > 0]
    if len(vary) < 2:
        return pd.DataFrame()
    z = usable[vary]
    scaled = (z - z.mean()) / z.std(ddof=0)
    scaled = pd.concat([usable[["inn"]].reset_index(drop=True), scaled.reset_index(drop=True)], axis=1)
    base = usable.reset_index(drop=True)
    for slice_name, col in SLICES:
        if col not in base.columns:
            continue
        labels = base[col].fillna("не заполнен").astype(str)
        counts = labels.value_counts()
        keep = counts[counts >= MIN_CLIENTS].index
        if len(keep) < 2:
            continue
        block = scaled.copy()
        block["_slice"] = labels.to_numpy()
        means = block.loc[block["_slice"].isin(keep)].groupby("_slice")[vary].mean()
        norms = np.linalg.norm(means.to_numpy(), axis=1)
        good = norms > 0
        means = means.loc[good]
        norms = norms[good]
        if len(means) < 2:
            continue
        unit = means.to_numpy() / norms[:, None]
        sim = unit @ unit.T
        names = list(means.index)
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                rows.append(
                    {
                        "slice_name": slice_name,
                        "group_a": names[i],
                        "group_b": names[j],
                        "n_a": int(counts[names[i]]),
                        "n_b": int(counts[names[j]]),
                        "cosine_similarity": float(sim[i, j]),
                    }
                )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["slice_name", "cosine_similarity"])

with imp:
    profile = imp.fetch("select * from sandbox_ai.tmp_shestopalov_te_profile")

profile_all = profile.copy()
before = len(profile)
profile = profile.loc[profile["mcc_code"].astype(str) != "не заполнен"].copy()
print(f"Клиентов в профиле: {before}. Без кода точки убрано: {before - len(profile)}. В расчёте: {len(profile)}.")
profile.head()


## Названия кодов

Рядом с цифровым кодом выводится название: MCC точки, ОКВЭД и регион регистрации по ОКАТО. Если полного кода ОКВЭД нет в справочнике, пишется ближайший верхний код, в который он входит. Слова в группе MCC (аптека, гостиница, бар) уже названия, их не переводим. Ячейка печатает коды точки, которых нет в справочнике.


In [ ]:
MCC_NAMES = {
    "742": "ветеринария",
    "763": "сельхозкооперативы",
    "780": "ландшафтный дизайн",
    "1520": "генподрядчики, строительство",
    "1711": "сантехника и отопление, подряд",
    "1731": "электромонтаж",
    "1799": "прочие строительные подрядчики",
    "4111": "пассажирский транспорт",
    "4121": "такси",
    "4131": "автобусные перевозки",
    "4214": "грузоперевозки",
    "4215": "курьерские услуги",
    "4722": "турагентства",
    "4784": "платные дороги",
    "4812": "телефоны и связь, оборудование",
    "4814": "услуги связи",
    "4816": "интернет, компьютерные сети",
    "4899": "кабельное и платное ТВ",
    "4900": "коммунальные услуги",
    "5013": "автозапчасти, опт",
    "5039": "стройматериалы, опт",
    "5044": "офисное оборудование",
    "5045": "компьютеры, опт",
    "5047": "медицинское оборудование",
    "5065": "электротовары, опт",
    "5072": "скобяные товары, опт",
    "5074": "сантехника, опт",
    "5085": "промышленные товары, опт",
    "5099": "товары длительного пользования, опт",
    "5111": "канцтовары, опт",
    "5122": "лекарства, опт",
    "5131": "ткани, опт",
    "5137": "спецодежда, опт",
    "5169": "химия, опт",
    "5172": "нефтепродукты, опт",
    "5192": "книги и печать, опт",
    "5193": "цветы и растения, опт",
    "5199": "прочие товары, опт",
    "5200": "товары для дома, крупный формат",
    "5211": "стройматериалы",
    "5231": "краски, стекло, обои",
    "5251": "скобяные товары, инструменты",
    "5261": "сад и огород",
    "5300": "оптовые клубы",
    "5309": "беспошлинные магазины",
    "5310": "дискаунтеры",
    "5311": "универмаги",
    "5331": "универсальные магазины",
    "5399": "товары смешанного ассортимента",
    "5411": "супермаркет",
    "5422": "мясо и рыба",
    "5441": "кондитерская",
    "5451": "молочный магазин",
    "5462": "пекарня",
    "5499": "прочие продуктовые",
    "5511": "автосалон",
    "5521": "подержанные автомобили",
    "5531": "автомагазин",
    "5532": "шины",
    "5533": "автозапчасти",
    "5541": "АЗС",
    "5542": "автоматическая АЗС",
    "5551": "лодки",
    "5571": "мотоциклы",
    "5599": "прочая автомототехника",
    "5611": "мужская одежда",
    "5621": "женская одежда",
    "5631": "женские аксессуары",
    "5641": "детская одежда",
    "5651": "одежда для всей семьи",
    "5655": "спортивная одежда",
    "5661": "обувь",
    "5681": "меха",
    "5691": "одежда мужская и женская",
    "5697": "ателье, пошив",
    "5699": "прочая одежда и аксессуары",
    "5712": "мебель",
    "5713": "напольные покрытия",
    "5714": "шторы и ткани для дома",
    "5719": "товары для дома",
    "5722": "бытовая техника",
    "5732": "электроника",
    "5733": "музыкальные инструменты",
    "5734": "программное обеспечение",
    "5735": "музыкальные записи",
    "5811": "кейтеринг",
    "5812": "ресторан",
    "5813": "бар",
    "5814": "фастфуд",
    "5815": "цифровые книги и фильмы",
    "5816": "цифровые игры",
    "5817": "цифровые приложения",
    "5818": "цифровые товары",
    "5912": "аптека",
    "5921": "алкоголь",
    "5931": "секонд-хенд",
    "5932": "антиквариат",
    "5933": "ломбард",
    "5940": "велосипеды",
    "5941": "спорттовары",
    "5942": "книжный магазин",
    "5943": "канцтовары",
    "5944": "ювелирный магазин, часы",
    "5945": "игрушки",
    "5946": "фототовары",
    "5947": "подарки и сувениры",
    "5948": "кожгалантерея",
    "5949": "ткани и швейные принадлежности",
    "5950": "посуда",
    "5970": "товары для творчества",
    "5971": "художественные галереи",
    "5973": "религиозные товары",
    "5975": "слуховые аппараты",
    "5976": "ортопедия",
    "5977": "косметика",
    "5983": "топливо: уголь, газ в баллонах",
    "5992": "цветы",
    "5993": "табак",
    "5994": "газеты и журналы",
    "5995": "зоотовары",
    "5996": "бассейны",
    "5998": "палатки и тенты",
    "5999": "прочие специализированные магазины",
    "6010": "выдача наличных в отделении",
    "6011": "банкомат",
    "6012": "финансовые услуги",
    "6051": "валюта, переводы",
    "6300": "страхование",
    "6513": "аренда недвижимости",
    "7011": "гостиница",
    "7032": "спортивные и детские лагеря",
    "7033": "кемпинги",
    "7210": "прачечные",
    "7216": "химчистка",
    "7221": "фотостудия",
    "7230": "парикмахерская, салон красоты",
    "7251": "ремонт обуви",
    "7261": "ритуальные услуги",
    "7277": "консультации, психологи",
    "7296": "прокат одежды",
    "7297": "массаж",
    "7298": "спа, оздоровление",
    "7299": "прочие бытовые услуги",
    "7311": "реклама",
    "7333": "фото и графический дизайн",
    "7338": "копирование и печать",
    "7342": "дезинсекция",
    "7349": "уборка",
    "7361": "кадровые агентства",
    "7372": "программирование",
    "7379": "ремонт компьютеров",
    "7392": "консалтинг",
    "7393": "охрана",
    "7394": "прокат оборудования",
    "7395": "фотолаборатория",
    "7399": "прочие бизнес-услуги",
    "7512": "прокат автомобилей",
    "7523": "парковка",
    "7531": "кузовной ремонт",
    "7534": "шиномонтаж",
    "7535": "покраска автомобилей",
    "7538": "автосервис",
    "7542": "автомойка",
    "7549": "эвакуатор",
    "7622": "ремонт электроники",
    "7623": "ремонт кондиционеров",
    "7629": "ремонт бытовой техники",
    "7631": "ремонт часов и ювелирных изделий",
    "7641": "перетяжка мебели",
    "7692": "сварка",
    "7699": "прочий ремонт",
    "7832": "кинотеатр",
    "7911": "танцевальная студия",
    "7922": "театр, билеты",
    "7929": "музыканты",
    "7932": "бильярд",
    "7933": "боулинг",
    "7941": "спортивные клубы, стадионы",
    "7991": "достопримечательности",
    "7993": "игровые автоматы",
    "7994": "компьютерный клуб",
    "7996": "парк развлечений",
    "7997": "фитнес, клубы",
    "7998": "зоопарк, аквариум",
    "7999": "прочие развлечения",
    "8011": "врачи",
    "8021": "стоматология",
    "8031": "остеопаты",
    "8041": "мануальная терапия",
    "8042": "окулисты",
    "8043": "оптика",
    "8049": "подологи",
    "8050": "сиделки, уход",
    "8062": "больница",
    "8071": "медицинская лаборатория",
    "8099": "прочие медицинские услуги",
    "8111": "юристы",
    "8211": "школы",
    "8220": "вузы",
    "8241": "заочное обучение",
    "8244": "бизнес-школы",
    "8249": "профобучение",
    "8299": "курсы, прочее образование",
    "8351": "детский сад",
    "8398": "благотворительность",
    "8641": "общественные организации",
    "8661": "религиозные организации",
    "8699": "членские организации",
    "8734": "лаборатории",
    "8911": "архитекторы",
    "8931": "бухгалтерия",
    "8999": "прочие профессиональные услуги",
    "9222": "штрафы",
    "9311": "налоги",
    "9399": "госуслуги",
    "9402": "почта",
}

OKVED_NAMES = {
    "01": "растениеводство и животноводство",
    "02": "лесоводство и лесозаготовки",
    "03": "рыболовство и рыбоводство",
    "05": "добыча угля",
    "06": "добыча нефти и газа",
    "07": "добыча металлических руд",
    "08": "добыча прочих полезных ископаемых",
    "09": "услуги в добыче",
    "10": "производство пищевых продуктов",
    "11": "производство напитков",
    "12": "производство табачных изделий",
    "13": "производство текстиля",
    "14": "производство одежды",
    "15": "производство кожи и изделий из кожи",
    "16": "обработка древесины",
    "17": "производство бумаги",
    "18": "полиграфия",
    "19": "кокс и нефтепродукты",
    "20": "химическое производство",
    "21": "производство лекарств",
    "22": "резиновые и пластмассовые изделия",
    "23": "неметаллическая минеральная продукция",
    "24": "металлургия",
    "25": "готовые металлические изделия",
    "26": "компьютеры, электроника, оптика",
    "27": "электрическое оборудование",
    "28": "машины и оборудование",
    "29": "автотранспортные средства",
    "30": "прочие транспортные средства",
    "31": "производство мебели",
    "32": "прочие готовые изделия",
    "33": "ремонт и монтаж машин",
    "35": "электроэнергия, газ, пар",
    "36": "водоснабжение",
    "37": "сточные воды",
    "38": "сбор и утилизация отходов",
    "39": "рекультивация",
    "41": "строительство зданий",
    "42": "строительство инженерных сооружений",
    "43": "специализированные строительные работы",
    "45": "торговля и ремонт автомобилей",
    "46": "оптовая торговля",
    "47": "розничная торговля",
    "49": "сухопутный транспорт",
    "50": "водный транспорт",
    "51": "воздушный транспорт",
    "52": "склады и вспомогательная транспортная деятельность",
    "53": "почта и курьеры",
    "55": "временное проживание",
    "56": "общепит",
    "58": "издательская деятельность",
    "59": "кино, видео, звукозапись",
    "60": "теле- и радиовещание",
    "61": "связь",
    "62": "разработка ПО",
    "63": "информационные технологии",
    "64": "финансовые услуги",
    "65": "страхование",
    "66": "вспомогательные финансовые услуги",
    "68": "операции с недвижимостью",
    "69": "право и бухучёт",
    "70": "управление, консалтинг",
    "71": "архитектура и инженерия",
    "72": "научные исследования",
    "73": "реклама и исследование рынка",
    "74": "прочая профессиональная деятельность",
    "75": "ветеринария",
    "77": "аренда и лизинг",
    "78": "подбор персонала",
    "79": "туристические агентства",
    "80": "охрана и расследования",
    "81": "обслуживание зданий",
    "82": "административно-хозяйственные услуги",
    "84": "госуправление",
    "85": "образование",
    "86": "здравоохранение",
    "87": "уход с проживанием",
    "88": "социальные услуги",
    "90": "творчество и искусство",
    "91": "библиотеки, музеи",
    "92": "азартные игры",
    "93": "спорт, отдых, развлечения",
    "94": "общественные организации",
    "95": "ремонт компьютеров и бытовых вещей",
    "96": "прочие личные услуги",
    "01.11": "зерновые и масличные",
    "01.11.1": "выращивание зерновых",
    "01.13": "овощи и корнеплоды",
    "01.41": "молочный скот",
    "01.45": "овцы и козы",
    "01.46": "свиньи",
    "01.47": "птица",
    "01.50": "смешанное сельское хозяйство",
    "10.11": "переработка мяса",
    "10.13": "мясные продукты",
    "10.51": "молочная продукция",
    "10.71": "хлеб и недолговечная выпечка",
    "10.89": "прочие пищевые продукты",
    "45.20": "ремонт автомобилей",
    "45.32": "розница автозапчастями",
    "46.90": "неспециализированный опт",
    "47.1": "неспециализированная розница",
    "47.11": "продовольственный магазин",
    "47.19": "прочий неспециализированный магазин",
    "47.2": "специализированная продуктовая розница",
    "47.21": "овощи и фрукты",
    "47.22": "мясо",
    "47.23": "рыба",
    "47.24": "хлеб и кондитерские изделия",
    "47.25": "напитки",
    "47.26": "табак",
    "47.29": "прочие продукты",
    "47.30": "АЗС, моторное топливо",
    "47.4": "компьютеры и связь",
    "47.5": "бытовые товары",
    "47.51": "текстиль",
    "47.52": "скобяные товары, краски, стекло",
    "47.53": "ковры и обои",
    "47.54": "бытовая электротехника",
    "47.59": "мебель и прочие бытовые товары",
    "47.6": "товары для досуга",
    "47.61": "книги",
    "47.62": "газеты и канцтовары",
    "47.64": "спорттовары",
    "47.65": "игрушки",
    "47.7": "прочие специализированные магазины",
    "47.71": "одежда",
    "47.72": "обувь и кожаные изделия",
    "47.73": "аптека",
    "47.74": "медицинские и ортопедические товары",
    "47.75": "косметика",
    "47.76": "цветы, семена, животные и корма",
    "47.76.2": "домашние животные и корма",
    "47.77": "часы и ювелирные изделия",
    "47.78": "прочие специализированные товары",
    "47.79": "подержанные товары",
    "47.8": "рынки и киоски",
    "47.81": "продукты на рынках и в киосках",
    "47.82": "одежда и обувь на рынках",
    "47.89": "прочие товары на рынках",
    "47.9": "розница вне магазинов",
    "47.91": "розница через интернет и по почте",
    "47.99": "прочая розница вне магазинов",
    "49.32": "такси",
    "49.41": "грузовые автоперевозки",
    "55.10": "гостиница",
    "55.20": "краткосрочное проживание",
    "56.10": "рестораны и доставка еды",
    "56.21": "кейтеринг",
    "56.29": "столовые",
    "56.30": "бары",
    "68.20": "аренда своей недвижимости",
    "68.31": "агентства недвижимости",
    "85.41": "дополнительное образование",
    "86.10": "больницы",
    "86.21": "общая врачебная практика",
    "86.22": "специальная врачебная практика",
    "86.23": "стоматология",
    "86.90": "прочая медицина",
    "93.11": "спортивные объекты",
    "93.13": "фитнес-центры",
    "93.29": "прочие развлечения",
    "96.01": "стирка и химчистка",
    "96.02": "парикмахерские и салоны красоты",
    "96.02.1": "парикмахерские",
    "96.02.2": "косметические услуги",
    "96.03": "ритуальные услуги",
    "96.04": "бани, сауны",
    "96.09": "прочие личные услуги",
}

REGION_NAMES = {
    "01": "Алтайский край",
    "03": "Краснодарский край",
    "04": "Красноярский край",
    "05": "Приморский край",
    "07": "Ставропольский край",
    "08": "Хабаровский край",
    "10": "Амурская обл.",
    "11": "Архангельская обл.",
    "12": "Астраханская обл.",
    "14": "Белгородская обл.",
    "15": "Брянская обл.",
    "17": "Владимирская обл.",
    "18": "Волгоградская обл.",
    "19": "Вологодская обл.",
    "20": "Воронежская обл.",
    "22": "Нижегородская обл.",
    "24": "Ивановская обл.",
    "25": "Иркутская обл.",
    "26": "Ингушетия",
    "27": "Калининградская обл.",
    "28": "Тверская обл.",
    "29": "Калужская обл.",
    "30": "Камчатский край",
    "32": "Кемеровская обл.",
    "33": "Кировская обл.",
    "34": "Костромская обл.",
    "35": "Крым",
    "36": "Самарская обл.",
    "37": "Курганская обл.",
    "38": "Курская обл.",
    "40": "Санкт-Петербург",
    "41": "Ленинградская обл.",
    "42": "Липецкая обл.",
    "44": "Магаданская обл.",
    "45": "Москва",
    "46": "Московская обл.",
    "47": "Мурманская обл.",
    "49": "Новгородская обл.",
    "50": "Новосибирская обл.",
    "52": "Омская обл.",
    "53": "Оренбургская обл.",
    "54": "Орловская обл.",
    "56": "Пензенская обл.",
    "57": "Пермский край",
    "58": "Псковская обл.",
    "60": "Ростовская обл.",
    "61": "Рязанская обл.",
    "63": "Саратовская обл.",
    "64": "Сахалинская обл.",
    "65": "Свердловская обл.",
    "66": "Смоленская обл.",
    "67": "Севастополь",
    "68": "Тамбовская обл.",
    "69": "Томская обл.",
    "70": "Тульская обл.",
    "71": "Тюменская обл.",
    "73": "Ульяновская обл.",
    "75": "Челябинская обл.",
    "76": "Забайкальский край",
    "77": "Чукотский АО",
    "78": "Ярославская обл.",
    "79": "Адыгея",
    "80": "Башкортостан",
    "81": "Бурятия",
    "82": "Дагестан",
    "83": "Кабардино-Балкария",
    "84": "Республика Алтай",
    "85": "Калмыкия",
    "86": "Карелия",
    "87": "Коми",
    "88": "Марий Эл",
    "89": "Мордовия",
    "90": "Северная Осетия",
    "91": "Карачаево-Черкесия",
    "92": "Татарстан",
    "93": "Тыва",
    "94": "Удмуртия",
    "95": "Хакасия",
    "96": "Чечня",
    "97": "Чувашия",
    "98": "Якутия",
    "99": "Еврейская АО",
}


def mcc_label(value):
    text = str(value).strip()
    if not text.isdigit():
        return text
    name = MCC_NAMES.get(text.lstrip("0") or "0")
    return f"{text} {name}" if name else f"{text} (нет в справочнике)"


def okved_label(value):
    text = str(value).strip()
    if text in ("", "не заполнен"):
        return text
    key = text
    while key:
        if key in OKVED_NAMES:
            if key == text:
                return f"{text} {OKVED_NAMES[key]}"
            return f"{text} (входит в {key} {OKVED_NAMES[key]})"
        key = key.rsplit(".", 1)[0] if "." in key else ""
    return f"{text} (нет в справочнике)"


def region_label(value):
    text = str(value).strip()
    name = REGION_NAMES.get(text)
    return f"{text} {name}" if name else text


SLICE_COLS = dict(SLICES)


def value_label(slice_name, value):
    col = SLICE_COLS.get(slice_name)
    text = str(value)
    if col in ("mcc_code", "mcc_group"):
        return mcc_label(text)
    if col == "slice_mcc_region":
        code, _, region = text.partition(" | ")
        return f"{mcc_label(code)} | {region_label(region)}"
    if col == "slice_mcc_size":
        code, _, size = text.partition(" | ")
        return f"{mcc_label(code)} | {size}"
    if col in ("okved", "okved_class"):
        return okved_label(text)
    return text


missing = sorted(
    {
        str(v)
        for v in profile["mcc_code"].astype(str).unique()
        if v.isdigit() and (v.lstrip("0") or "0") not in MCC_NAMES
    }
)
print("Кодов точки без названия в справочнике:", len(missing), missing[:30])


## Численность групп

`достаточно` — от 100 клиентов, `на грани` — от 30 до 99, `мало` — меньше 30. Тесты распределений и косинус считаются только там, где клиентов не меньше 30, и в дополнении группы тоже не меньше 30.


In [ ]:
coverage = coverage_table(profile)
coverage.insert(2, "name", [value_label(s, v) for s, v in zip(coverage["slice_name"], coverage["slice_value"])])
summary = (
    coverage.groupby(["slice_name", "enough"], as_index=False)
    .agg(groups=("slice_value", "count"), clients=("client_cnt", "sum"))
    .sort_values(["slice_name", "enough"])
)
display(summary)
usable = coverage.loc[coverage["enough"] != "мало"].copy()
print(f"Групп, где клиентов не меньше 30: {len(usable)}.")
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 80):
    display(usable[["slice_name", "name", "client_cnt", "client_share", "enough"]])


## 1. Сравнение распределений

Каждая группа сравнивается с остальными клиентами того же разреза. Колмогоров–Смирнов и Манн–Уитни показывают, различаются ли распределения. Wasserstein — насколько далеко сдвинуты суммы и количества. Jensen–Shannon — насколько различается форма. p-значения Колмогорова–Смирнова и Манна–Уитни поправлены по Бенджамини–Хохбергу внутри разреза и метрики.

Ниже сначала строки, где после поправки p < 0.05, с самым большим сдвигом Wasserstein. Затем хи-квадрат и V Крамера: связаны ли группа MCC и регион, группа MCC и размер бизнеса, класс ОКВЭД и регион, класс ОКВЭД и размер. Это связь категорий, не сравнение сумм чека.


In [ ]:
tests = compare_distributions(profile)
if tests.empty:
    print("Нет группы, где и она сама, и остальные имеют не меньше 30 клиентов.")
else:
    tests.insert(2, "name", [value_label(s, v) for s, v in zip(tests["slice_name"], tests["slice_value"])])
    strong = tests.loc[tests["ks_pvalue_bh"] < 0.05].sort_values("wasserstein", ascending=False)
    print(f"Сравнений: {len(tests)}. После поправки p < 0.05: {len(strong)}.")
    cols = ["slice_name", "name", "metric", "n_group", "n_rest", "wasserstein", "jensen_shannon", "ks_pvalue_bh", "mannwhitney_pvalue_bh"]
    with pd.option_context("display.max_colwidth", 80):
        display(strong[cols].head(30))

assoc = association_tests(profile)
if assoc.empty:
    print("Хи-квадрат не посчитан: нет двух категорий с достаточным числом клиентов.")
else:
    display(assoc)


## 2. Нормализация и косинус

Поля портрета приводятся к одному масштабу (z-нормализация по всем клиентам расчёта). Для каждой группы берётся средний портрет. Косинус близок к 1, когда средние портреты похожи, и к 0, когда они расходятся. Сравниваются группы, а не отдельные клиенты. В пару попадают только группы от 30 клиентов.


In [ ]:
cosine = cosine_by_slice(profile)
if cosine.empty:
    print("Косинус не посчитан: в разрезе нет двух групп от 30 клиентов.")
else:
    cosine["name_a"] = [value_label(s, v) for s, v in zip(cosine["slice_name"], cosine["group_a"])]
    cosine["name_b"] = [value_label(s, v) for s, v in zip(cosine["slice_name"], cosine["group_b"])]
    pieces = []
    for slice_name, part in cosine.groupby("slice_name"):
        ordered = part.sort_values("cosine_similarity")
        far = ordered.head(5).assign(pair="далёкие")
        near = ordered.tail(5).assign(pair="близкие")
        pieces.append(pd.concat([far, near], ignore_index=True))
    show = pd.concat(pieces, ignore_index=True)
    with pd.option_context("display.max_rows", 200, "display.max_colwidth", 80):
        display(show[["slice_name", "pair", "name_a", "name_b", "n_a", "n_b", "cosine_similarity"]])


## Выгрузка в Excel

Все таблицы сохраняются в один файл рядом с тетрадкой: `te_distributions_2026_10.xlsx`. Листы:

- `регионы` — все регионы регистрации по ОКАТО: клиенты, операции, оборот, доли, место; отметка, входит ли регион в пятёрку;
- `mcc` — все коды точки: клиенты, операции, оборот, доли, место; в какую группу MCC попал код и остался ли он отдельной группой;
- `численность` — все группы всех разрезов, сколько клиентов и годится ли группа для тестов;
- `тесты` — все сравнения «группа против остальных» по каждому полю портрета, не только значимые;
- `хи_квадрат` — связь пар подписей;
- `косинус` — все пары групп внутри разреза, не только по пять крайних.

Если на сервере нет библиотеки для записи xlsx, те же таблицы пишутся в четыре csv с такими же именами.

На листах `тесты` и `хи_квадрат` рядом с точными p-значениями есть читаемые: всё, что меньше 0,001, пишется как `< 0,001`. На листе `тесты` колонка `вывод` говорит словами, различается ли группа, по поправленным p.


In [ ]:
from pathlib import Path


def p_text(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "< 0,001"
    return f"{p:.3f}".replace(".", ",")


def verdict(ks_p, mw_p):
    ks_yes = ks_p < 0.05
    mw_yes = mw_p < 0.05
    if ks_yes and mw_yes:
        return "различается"
    if ks_yes:
        return "различается форма, сдвига нет"
    if mw_yes:
        return "есть сдвиг, форма похожа"
    return "не различается"


tests_out = tests.copy()
if not tests_out.empty:
    tests_out["ks_p_читаемо"] = tests_out["ks_pvalue_bh"].map(p_text)
    tests_out["mw_p_читаемо"] = tests_out["mannwhitney_pvalue_bh"].map(p_text)
    tests_out["вывод"] = [verdict(k, m) for k, m in zip(tests_out["ks_pvalue_bh"], tests_out["mannwhitney_pvalue_bh"])]
assoc_out = assoc.copy()
if not assoc_out.empty:
    assoc_out["p_читаемо"] = assoc_out["pvalue"].map(p_text)


def rank_table(frame, col, label_fn):
    work = frame.copy()
    work["код"] = work[col].fillna("не заполнен").astype(str)
    work["tx_count"] = pd.to_numeric(work["tx_count"], errors="coerce")
    work["turnover"] = pd.to_numeric(work["turnover"], errors="coerce")
    out = work.groupby("код", as_index=False).agg(
        клиентов=("inn", "nunique"),
        операций=("tx_count", "sum"),
        оборот=("turnover", "sum"),
    )
    out.insert(1, "название", out["код"].map(label_fn))
    out["доля_клиентов"] = out["клиентов"] / out["клиентов"].sum()
    out["доля_оборота"] = out["оборот"] / out["оборот"].sum()
    out["место_по_обороту"] = out["оборот"].rank(ascending=False, method="min").astype(int)
    out = out.sort_values(["клиентов", "оборот"], ascending=False).reset_index(drop=True)
    out.insert(0, "место_по_клиентам", range(1, len(out) + 1))
    return out


regions_out = rank_table(profile_all, "region", lambda c: REGION_NAMES.get(c, ""))
top5 = set(profile_all.loc[profile_all["region_group"].astype(str) != "прочие", "region"].astype(str))
regions_out["в_топ_5"] = regions_out["код"].isin(top5).map({True: "да", False: "нет, в «прочих»"})

mcc_out = rank_table(profile_all, "mcc_code", lambda c: MCC_NAMES.get(c.lstrip("0") or "0", ""))
groups = profile_all.assign(_c=profile_all["mcc_code"].astype(str)).groupby("_c")["mcc_group"].first()
mcc_out["группа_mcc"] = mcc_out["код"].map(groups)
mcc_out["отдельная_группа"] = (mcc_out["группа_mcc"].astype(str) == mcc_out["код"]).map({True: "да", False: "нет, склеен"})

print("Крупнейшие регионы по числу клиентов:")
display(regions_out.head(10))
print("Крупнейшие коды точки по числу клиентов:")
display(mcc_out.head(15))

sheets = {
    "регионы": regions_out,
    "mcc": mcc_out,
    "численность": coverage,
    "тесты": tests_out,
    "хи_квадрат": assoc_out,
    "косинус": cosine,
}
out_xlsx = Path.cwd() / "te_distributions_2026_10.xlsx"
try:
    with pd.ExcelWriter(out_xlsx) as writer:
        for sheet, frame in sheets.items():
            frame.to_excel(writer, sheet_name=sheet, index=False)
    print("Сохранено:", out_xlsx)
except ImportError:
    for sheet, frame in sheets.items():
        out_csv = Path.cwd() / f"te_distributions_2026_10_{sheet}.csv"
        frame.to_csv(out_csv, index=False, encoding="utf-8-sig", sep=";")
        print("Сохранено:", out_csv)
